# 📓 Notebook 05 — Model Evaluation
## G19 Facial Emotion Recognition | CSE276

---

**Purpose of this notebook:**
- Load the best trained model.
- Evaluate on the held-out **test set** (PrivateTest).
- Generate and save:
  - Per-class classification report (precision, recall, F1)
  - Confusion matrix heatmap
  - Per-class accuracy bar chart
  - Training history curves (if available)
- Interpret results honestly, including weaknesses.

> **Prerequisite:** Notebook 04 completed and `models/best_model.keras` exists.

In [ ]:
import sys, os
sys.path.insert(0, '/content/G19_Facial_Emotion_Recognition')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.data_loader   import load_fer2013
from src.preprocessing import apply_clahe_batch
from src.evaluation    import evaluate_model, plot_training_history
from src.utils         import set_random_seeds, load_model_safe

set_random_seeds(42)

DATASET_CSV  = '/content/G19_Facial_Emotion_Recognition/dataset/fer2013.csv'
MODEL_PATH   = '/content/G19_Facial_Emotion_Recognition/models/best_model.keras'
RESULTS_DIR  = '/content/G19_Facial_Emotion_Recognition/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

## 1. Load Dataset and Model

In [ ]:
# PrivateTest is evaluated ONLY here for final unbiased test reporting.
_, _, _, _, X_test, y_test = load_fer2013(DATASET_CSV)

# Must match the USE_CLAHE setting used during training (False for baseline)
USE_CLAHE = False
if USE_CLAHE:
    X_test = apply_clahe_batch(X_test)

print(f'Test set (PrivateTest): {X_test.shape} | USE_CLAHE={USE_CLAHE}')

model = load_model_safe(MODEL_PATH)
model.summary()

## 2. Full Evaluation on Test Set

In [ ]:
# This runs model.evaluate, prints classification report, and saves plots
results = evaluate_model(model, X_test, y_test)

print(f"\n--- Final Test Results ---")
print(f"Test Loss     : {results['test_loss']:.4f}")
print(f"Test Accuracy : {results['test_acc'] * 100:.2f}%")

## 3. Display Confusion Matrix

In [ ]:
img = Image.open(f'{RESULTS_DIR}/confusion_matrix.png')
plt.figure(figsize=(9, 7))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix (Test Set)', fontsize=13)
plt.show()

## 4. Display Per-Class Accuracy

In [ ]:
img = Image.open(f'{RESULTS_DIR}/per_class_accuracy.png')
plt.figure(figsize=(9, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. Examine Misclassified Examples

In [ ]:
import numpy as np
from src.data_loader import EMOTION_LABELS

# Get predictions
y_probs = model.predict(X_test, verbose=0)
y_pred  = np.argmax(y_probs, axis=1)

# Find misclassified indices
wrong_indices = np.where(y_pred != y_test)[0]
print(f'Total misclassified: {len(wrong_indices)} / {len(y_test)} ({len(wrong_indices)/len(y_test)*100:.1f}%)')

# Display up to 15 misclassified examples
n_show = min(15, len(wrong_indices))
sample_wrong = np.random.choice(wrong_indices, size=n_show, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(13, 8))
axes = axes.flatten()

for i, idx in enumerate(sample_wrong):
    axes[i].imshow(X_test[idx, :, :, 0], cmap='gray')
    true_label = EMOTION_LABELS[y_test[idx]]
    pred_label = EMOTION_LABELS[y_pred[idx]]
    conf       = y_probs[idx, y_pred[idx]]
    axes[i].set_title(
        f'True: {true_label}\nPred: {pred_label} ({conf:.0%})',
        fontsize=8, color='red'
    )
    axes[i].axis('off')

plt.suptitle('Misclassified Examples', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/misclassified_examples.png', dpi=120)
plt.show()

## 6. Results Summary

> ⚠️ **Fill this section in after running all cells with real results.**
> Do NOT invent numbers.

| Metric | Value |
|---|---|
| Test accuracy | _(fill after running)_ |
| Test loss | _(fill after running)_ |
| Highest F1 class | _(fill after running)_ |
| Lowest F1 class | _(fill after running)_ |
| Most confused pair | _(fill after running)_ |

### Observations
- _(What worked well?)_
- _(Which emotions are hardest to classify and why?)_
- _(Does the class imbalance affect specific classes?)_

### Limitations
- _(Performance gaps due to dataset size / quality)_
- _(Challenges with minority classes)_

**See `documentation/limitations_and_ethics.md` for a full discussion.**